In [1]:
import os
import glob
import random
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
base_project_path = "/Users/vihangainduwara/Desktop/MelanoDetect"

ham_base_path = os.path.join(base_project_path, "data", "ham10000")
metadata_path = os.path.join(ham_base_path, "HAM10000_metadata.csv")
images_part1 = os.path.join(ham_base_path, "HAM10000_images_part_1")
images_part2 = os.path.join(ham_base_path, "HAM10000_images_part_2")

invalid_raw_dir = os.path.join(base_project_path, "data", "input_validator_raw", "invalid_other")
split_base_dir = os.path.join(base_project_path, "data", "input_validator_split")

print("Metadata exists:", os.path.exists(metadata_path))
print("Invalid raw dir exists:", os.path.exists(invalid_raw_dir))

Metadata exists: True
Invalid raw dir exists: True


In [3]:
base_project_path = "/Users/vihangainduwara/Desktop/MelanoDetect"

ham_base_path = os.path.join(base_project_path, "data", "ham10000")
metadata_path = os.path.join(ham_base_path, "HAM10000_metadata.csv")
images_part1 = os.path.join(ham_base_path, "HAM10000_images_part_1")
images_part2 = os.path.join(ham_base_path, "HAM10000_images_part_2")

invalid_raw_dir = os.path.join(base_project_path, "data", "input_validator_raw", "invalid_other")
split_base_dir = os.path.join(base_project_path, "data", "input_validator_split")

print("Metadata exists:", os.path.exists(metadata_path))
print("Invalid raw dir exists:", os.path.exists(invalid_raw_dir))

Metadata exists: True
Invalid raw dir exists: True


In [4]:
df = pd.read_csv(metadata_path)

image_paths_part1 = glob.glob(os.path.join(images_part1, "*.jpg"))
image_paths_part2 = glob.glob(os.path.join(images_part2, "*.jpg"))
all_image_paths = image_paths_part1 + image_paths_part2

image_path_dict = {
    os.path.splitext(os.path.basename(path))[0]: path
    for path in all_image_paths
}

df["image_path"] = df["image_id"].map(image_path_dict)

valid_lesion_paths = df["image_path"].dropna().tolist()

print("Total valid lesion images:", len(valid_lesion_paths))

Total valid lesion images: 10015


In [5]:
invalid_extensions = ["*.jpg", "*.jpeg", "*.png", "*.webp"]

invalid_paths = []
for ext in invalid_extensions:
    invalid_paths.extend(glob.glob(os.path.join(invalid_raw_dir, ext)))

print("Total invalid images found:", len(invalid_paths))

Total invalid images found: 12958


In [6]:
random.seed(42)

if len(invalid_paths) == 0:
    raise ValueError("No invalid images found. Put random non-lesion images into input_validator_raw/invalid_other first.")

sampled_valid_paths = random.sample(valid_lesion_paths, min(len(valid_lesion_paths), len(invalid_paths)))

print("Using valid lesion images:", len(sampled_valid_paths))
print("Using invalid images:", len(invalid_paths))

Using valid lesion images: 10015
Using invalid images: 12958


In [7]:
valid_train, valid_temp = train_test_split(sampled_valid_paths, test_size=0.30, random_state=42)
valid_val, valid_test = train_test_split(valid_temp, test_size=0.50, random_state=42)

invalid_train, invalid_temp = train_test_split(invalid_paths, test_size=0.30, random_state=42)
invalid_val, invalid_test = train_test_split(invalid_temp, test_size=0.50, random_state=42)

print("VALID SPLIT")
print(len(valid_train), len(valid_val), len(valid_test))

print("INVALID SPLIT")
print(len(invalid_train), len(invalid_val), len(invalid_test))

VALID SPLIT
7010 1502 1503
INVALID SPLIT
9070 1944 1944


In [8]:
subdirs = [
    "train/valid_lesion",
    "train/invalid_other",
    "val/valid_lesion",
    "val/invalid_other",
    "test/valid_lesion",
    "test/invalid_other",
]

if os.path.exists(split_base_dir):
    shutil.rmtree(split_base_dir)

for subdir in subdirs:
    os.makedirs(os.path.join(split_base_dir, subdir), exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [9]:
def copy_files(file_list, destination_folder):
    for file_path in file_list:
        filename = os.path.basename(file_path)
        shutil.copy(file_path, os.path.join(destination_folder, filename))

copy_files(valid_train, os.path.join(split_base_dir, "train", "valid_lesion"))
copy_files(valid_val, os.path.join(split_base_dir, "val", "valid_lesion"))
copy_files(valid_test, os.path.join(split_base_dir, "test", "valid_lesion"))

copy_files(invalid_train, os.path.join(split_base_dir, "train", "invalid_other"))
copy_files(invalid_val, os.path.join(split_base_dir, "val", "invalid_other"))
copy_files(invalid_test, os.path.join(split_base_dir, "test", "invalid_other"))

print("Image copy completed.")

Image copy completed.


In [10]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

VAL_IMG_SIZE = (224, 224)
VAL_BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rotation_range=8,
    width_shift_range=0.03,
    height_shift_range=0.03,
    zoom_range=0.05,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator()

train_generator = train_datagen.flow_from_directory(
    os.path.join(split_base_dir, "train"),
    target_size=VAL_IMG_SIZE,
    batch_size=VAL_BATCH_SIZE,
    class_mode="binary",
    shuffle=True
)

val_generator = test_datagen.flow_from_directory(
    os.path.join(split_base_dir, "val"),
    target_size=VAL_IMG_SIZE,
    batch_size=VAL_BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    os.path.join(split_base_dir, "test"),
    target_size=VAL_IMG_SIZE,
    batch_size=VAL_BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

Found 16080 images belonging to 2 classes.
Found 3446 images belonging to 2 classes.
Found 3447 images belonging to 2 classes.


In [11]:
print(train_generator.class_indices)

{'invalid_other': 0, 'valid_lesion': 1}


In [12]:
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

validator_base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

validator_base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = validator_base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

validator_model = tf.keras.Model(inputs, outputs)

validator_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Validator model created.")

Validator model created.


In [13]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

validator_model_path = os.path.join(base_project_path, "models", "lesion_input_validator.keras")

validator_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        filepath=validator_model_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    )
]

print(validator_model_path)

/Users/vihangainduwara/Desktop/MelanoDetect/models/lesion_input_validator.keras


In [14]:
validator_history = validator_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=8,
    callbacks=validator_callbacks
)

Epoch 1/8
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 0.9829 - loss: 0.0765
Epoch 1: val_loss improved from None to 0.00689, saving model to /Users/vihangainduwara/Desktop/MelanoDetect/models/lesion_input_validator.keras
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 172s 168ms/step - accuracy: 0.9951 - loss: 0.0274 - val_accuracy: 0.9980 - val_loss: 0.0069 - learning_rate: 0.0010
Epoch 2/8
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.9989 - loss: 0.0047
Epoch 2: val_loss improved from 0.00689 to 0.00297, saving model to /Users/vihangainduwara/Desktop/MelanoDetect/models/lesion_input_validator.keras
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 165s 164ms/step - accuracy: 0.9993 - loss: 0.0038 - val_accuracy: 0.9991 - val_loss: 0.0030 - learning_rate: 0.0010
Epoch 3/8
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.9999 - loss: 0.0021
Epoch 3: val_loss improved from 0.00297 to 0.00197, saving model to /Users/vihangainduwara/Desktop/MelanoDetect/models/lesion_input_validator.kera

In [15]:
best_validator_model = tf.keras.models.load_model(validator_model_path)

val_test_loss, val_test_acc = best_validator_model.evaluate(test_generator)
print("Validator Test Loss:", val_test_loss)
print("Validator Test Accuracy:", val_test_acc)

216/216 ━━━━━━━━━━━━━━━━━━━━ 32s 143ms/step - accuracy: 1.0000 - loss: 5.7926e-04
Validator Test Loss: 0.0005792579613626003
Validator Test Accuracy: 1.0


In [16]:
import numpy as np

test_generator.reset()
validator_pred_probs = best_validator_model.predict(test_generator, verbose=1).flatten()
validator_true_labels = test_generator.classes

print("Total predictions:", len(validator_pred_probs))
print("Total true labels:", len(validator_true_labels))

216/216 ━━━━━━━━━━━━━━━━━━━━ 34s 153ms/step
Total predictions: 3447
Total true labels: 3447


In [17]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import pandas as pd

thresholds = [0.50, 0.60, 0.70, 0.80, 0.90]
results = []

for t in thresholds:
    preds = (validator_pred_probs >= t).astype(int)

    acc = accuracy_score(validator_true_labels, preds)
    precision = precision_score(validator_true_labels, preds)
    recall = recall_score(validator_true_labels, preds)
    f1 = f1_score(validator_true_labels, preds)

    results.append({
        "threshold": t,
        "accuracy": acc,
        "precision_valid": precision,
        "recall_valid": recall,
        "f1_valid": f1
    })

validator_threshold_df = pd.DataFrame(results)
validator_threshold_df.round(3)

,threshold,accuracy,precision_valid,recall_valid,f1_valid
0,0.5,1.0,1.0,1.0,1.0
1,0.6,1.0,1.0,1.0,1.0
2,0.7,1.0,1.0,1.0,1.0
3,0.8,1.0,1.0,1.0,1.0
4,0.9,1.0,1.0,1.0,1.0


In [18]:
print("FINAL VALIDATOR MODEL SUMMARY")
print("-----------------------------")
print("Validator model: lesion_input_validator.keras")
print("Purpose: Reject random / unsupported images before benign-malignant classification")
print("Classes: valid_lesion vs invalid_other")
print("Selected validator threshold: 0.80")
print("Main classifier used after validation: ham10000_efficientnetb0_big260_ft10.keras")
print("Main classifier threshold: 0.50")
print("Main classifier highest accuracy: 85.7%")

FINAL VALIDATOR MODEL SUMMARY
-----------------------------
Validator model: lesion_input_validator.keras
Purpose: Reject random / unsupported images before benign-malignant classification
Classes: valid_lesion vs invalid_other
Selected validator threshold: 0.80
Main classifier used after validation: ham10000_efficientnetb0_big260_ft10.keras
Main classifier threshold: 0.50
Main classifier highest accuracy: 85.7%


In [19]:
import os

model_files = [
    "/Users/vihangainduwara/Desktop/MelanoDetect/models/lesion_input_validator.keras",
    "/Users/vihangainduwara/Desktop/MelanoDetect/models/ham10000_efficientnetb0_big260_ft10.keras"
]

for path in model_files:
    print(path, "->", os.path.exists(path))

print("\nFINAL VALIDATOR + CLASSIFIER SUMMARY")
print("------------------------------------")
print("Validator model: lesion_input_validator.keras")
print("Validator threshold: 0.80")
print("Final classifier: ham10000_efficientnetb0_big260_ft10.keras")
print("Classifier threshold: 0.50")
print("Classifier highest accuracy: 85.7%")

/Users/vihangainduwara/Desktop/MelanoDetect/models/lesion_input_validator.keras -> True
/Users/vihangainduwara/Desktop/MelanoDetect/models/ham10000_efficientnetb0_big260_ft10.keras -> True

FINAL VALIDATOR + CLASSIFIER SUMMARY
------------------------------------
Validator model: lesion_input_validator.keras
Validator threshold: 0.80
Final classifier: ham10000_efficientnetb0_big260_ft10.keras
Classifier threshold: 0.50
Classifier highest accuracy: 85.7%


In [20]:
summary_text = """
FINAL VALIDATOR + CLASSIFIER SUMMARY
------------------------------------
Validator model: lesion_input_validator.keras
Validator threshold: 0.80
Purpose: Reject random / unsupported images before classification

Final classifier: ham10000_efficientnetb0_big260_ft10.keras
Classifier threshold: 0.50
Highest accuracy: 85.7%
"""

with open("/Users/vihangainduwara/Desktop/MelanoDetect/docs/final_validator_summary.txt", "w") as f:
    f.write(summary_text)

print("Validator summary saved successfully.")

Validator summary saved successfully.
